In [33]:
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, random_split, TensorDataset, Subset
from torchvision import datasets, transforms
import os
import random
import numpy as np

SEED = 0
torch.manual_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
g = torch.Generator()
g.manual_seed(0)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'  # or ':16:8'


################################
#     RESTART     RUNTIME      #
################################
from ssonn.metrics.nonlinearity_metrics import *
from ssonn.metrics.edge_finder import *
# from ssonn.metrics.train_metrics import *
from ssonn.train.train import *

In [34]:
class SimpleFCN(nn.Module):
    def __init__(self, input_size=28 * 28, hidden_size=16):
        super(SimpleFCN, self).__init__()
        self.fc0 = nn.Linear(input_size, 10)
        # self.fc1 = nn.Linear(hidden_size, 10)
        # self.act = nn.ReLU()

    def forward(self, x):
        x = self.fc0(x)
        return x

In [35]:
hyperparams = {
    "num_epochs": 32,
    "batch_size": 256,
    "edge_importance_metric": AbsGradientEdgeMetric(nn.CrossEntropyLoss()), # replace to default
    "edge_score_aggregation": "mean",
    "expansion_thresholds": {"fc0": 0.4},
    "pruning_thresholds": {"fc0": 1},
    "plateau_threshold": 0.05,
    "min_epochs_between_expansions": 8,
    # "min_epochs_between_expansions": 10_000,
    "plateau_window_size": 5,
    "learning_rate": 1e-3,
    "prune_after_epochs": 4,
    "task_type": "classification",
    "start_fully_connected": False,
    "max_new_edges_per_expansion": 2000,
    "weight_decay": 1e-4,
    "diff_threshold": 0.05
}

In [36]:
# Dataset and Dataloader
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.view(-1))
])

# Load dataset and split into train/validation sets
dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_dataset, val_dataset = random_split(dataset, [0.8, 0.2], generator=g)

# Новая логика: finder — доля train для расширения (100% = полный train)
finder_fraction = 1.0
n_finder = int(finder_fraction * len(train_dataset))
finder_dataset = Subset(train_dataset.dataset, train_dataset.indices[:n_finder])

batch_size = hyperparams['batch_size']
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, generator=g)
finder_loader = DataLoader(finder_dataset, batch_size=batch_size, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

In [37]:
torch.manual_seed(SEED)
model = SimpleFCN()
model = convert_to_scaled_net(model, layers_to_convert=[model.fc0], device=device)

Converting layer 'fc0' to ScaledNet...


In [38]:
# import wandb
# wandb.login()

In [39]:
# wandb.finish()
# run = wandb.init(
#     project="self-expanding-nets",
#     name=f"name",
#     config=hyperparams
# )

In [40]:
from comet_ml import start
from comet_ml.integration.pytorch import log_model
experiment = start(
    api_key=os.environ.get("COMET_API_KEY"),
    workspace=os.environ.get("COMET_WORKSPACE"),
    project_name="mnist",
)
experiment.log_parameters(hyperparams)
experiment.set_name("mnist origin")

COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: sklearn, torch.
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/shi-van/split-data/1a26c22bf77d488b8dbc518b6335f388



In [41]:
criterion = nn.CrossEntropyLoss()

def optimizer_factory(m):
    return optim.Adam(m.parameters(), lr=hyperparams['learning_rate'], weight_decay=hyperparams['weight_decay'])

trained_model = train_ssonn(
    model,
    train_loader,
    val_loader,
    finder_loader,
    criterion,
    optimizer_factory,
    hyperparams,
    device,
    experiment,
    True,
)

--- Starting SSONN Training ---


Epoch 1/32 | T-Loss: 0.9389 | V-Loss: 0.5425 | Met: 0.8680 | Params: 7850/7850


Epoch 2/32 | T-Loss: 0.4632 | V-Loss: 0.4157 | Met: 0.8902 | Params: 7850/7850


Epoch 3/32 | T-Loss: 0.3853 | V-Loss: 0.3680 | Met: 0.9014 | Params: 7850/7850


Epoch 4/32 | T-Loss: 0.3503 | V-Loss: 0.3424 | Met: 0.9075 | Params: 7850/7850


Epoch 5/32 | T-Loss: 0.3293 | V-Loss: 0.3269 | Met: 0.9108 | Params: 7850/7850


... (23 outputs left out)

Epoch 29/32 | T-Loss: 0.0171 | V-Loss: 0.0725 | Met: 0.9794 | Params: 1526825/1546175


Epoch 30/32 | T-Loss: 0.0139 | V-Loss: 0.0714 | Met: 0.9795 | Params: 1526825/1546175


Epoch 31/32 | T-Loss: 0.0135 | V-Loss: 0.0689 | Met: 0.9810 | Params: 1526825/1546175


Epoch 32/32 | T-Loss: 0.0141 | V-Loss: 0.0718 | Met: 0.9792 | Params: 1526825/1546175
--- Training Finished ---


In [42]:
experiment.end()

COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : mnist origin
COMET INFO:     url                   : https://www.comet.com/shi-van/split-data/1a26c22bf77d488b8dbc518b6335f388
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     expansion event    : 1935
COMET INFO:     lr [32]            : (0.0005, 0.001)
COMET INFO:     params amount [32] : (7850, 1526825)
COMET INFO:     sparsity [32]      : (0.0, 0.013766229566510857)
COMET INFO:     total_params [32]  : (7850, 1546175)
COMET INFO:     train loss [32]    : (0.013530073816234127, 0.9388579451974403)
COMET INFO:     val accuracy [32]  : (0.868, 0.981)
COMET INFO:     val loss [32]      : (0.06893567867735599, 0.5425162251959456)
COMET INFO: 

In [43]:
torch.save(model, 'mnist.pt')